# ALFWorld episode history analysis

Use this notebook to find, compare, and inspect individual episodes in `history.json`. The sample file is a JSON array with one record per episode and a nested `steps` list.

**Quick start**

1. Run all cells once.
2. Edit `EPISODE_IDS` and the optional filters in **Select episodes**.
3. Re-run from that cell downward.
4. Use `show_episode(...)` for a readable trajectory and `search_episodes(...)` to locate relevant episodes.

The loader treats every string in the history file as data. It never evaluates decisions, actions, or observations as code.

In [ ]:
from __future__ import annotations

import json
import os
import re
from collections import Counter
from pathlib import Path
from typing import Any, Iterable

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_colwidth", 140)

# Override this cell or set the HISTORY_PATH environment variable when the file moves.
PATH_CANDIDATES = [
    Path(os.environ["HISTORY_PATH"]).expanduser() if os.environ.get("HISTORY_PATH") else None,
    Path("/Users/yixiang/Downloads/history.json"),
    Path.cwd() / "history.json",
    Path.cwd().parent / "history.json",
]
HISTORY_PATH = next((path for path in PATH_CANDIDATES if path and path.is_file()), None)
if HISTORY_PATH is None:
    raise FileNotFoundError("Set HISTORY_PATH to the location of history.json")

print(f"Using {HISTORY_PATH.resolve()}")

## Load and normalize

Both a JSON array and JSON Lines (`.jsonl`) are accepted. The nested data is normalized into three tables: one row per episode, one row per step, and one row per sampled candidate.

In [ ]:
def load_history(path: Path) -> list[dict[str, Any]]:
    text = path.read_text(encoding="utf-8")
    try:
        payload = json.loads(text)
    except json.JSONDecodeError:
        payload = [json.loads(line) for line in text.splitlines() if line.strip()]

    if not isinstance(payload, list) or not all(isinstance(row, dict) for row in payload):
        raise ValueError("Expected a JSON array or JSONL stream of episode objects")
    return payload


episodes = load_history(HISTORY_PATH)
required_episode_fields = {"episode", "steps", "success"}
for position, record in enumerate(episodes):
    missing = required_episode_fields - record.keys()
    if missing:
        raise ValueError(f"Episode record {position} is missing {sorted(missing)}")
    if not isinstance(record["steps"], list):
        raise ValueError(f"Episode record {position} has a non-list 'steps' value")

episode_ids = [record["episode"] for record in episodes]
if len(episode_ids) != len(set(episode_ids)):
    raise ValueError("Episode IDs must be unique")

episode_rows: list[dict[str, Any]] = []
step_rows: list[dict[str, Any]] = []
candidate_rows: list[dict[str, Any]] = []

for episode in episodes:
    steps = episode.get("steps", [])
    episode_rows.append({
        "episode": episode.get("episode"),
        "task_key": episode.get("task_key"),
        "success": bool(episode.get("success")),
        "n_steps": len(steps),
        "task_description": episode.get("task_description"),
        "gamefile": episode.get("gamefile"),
        "initial_observation": episode.get("initial_observation"),
        "fallback_steps": sum(bool(step.get("fallback_used")) for step in steps),
        "retry_count": sum(int(step.get("empty_action_retries") or 0) for step in steps),
        "sample_attempts": sum(int(step.get("sample_attempts") or 0) for step in steps),
    })

    for step_position, step in enumerate(steps):
        candidates = step.get("candidates") or []
        step_rows.append({
            "episode": episode.get("episode"),
            "task_key": episode.get("task_key"),
            "success": bool(episode.get("success")),
            "step": step.get("step", step_position),
            "action": step.get("action_taken", step.get("action")),
            "model_action": step.get("selected_model_action"),
            "decision": step.get("selected_model_decision", step.get("decision")),
            "observation": step.get("observation"),
            "selected_sample": step.get("selected_sample"),
            "sample_attempts": step.get("sample_attempts"),
            "empty_action_retries": step.get("empty_action_retries"),
            "fallback_used": bool(step.get("fallback_used")),
            "fallback_reason": step.get("fallback_reason"),
            "fallback_policy": step.get("fallback_policy"),
            "n_candidates": len(candidates),
        })

        for candidate_position, candidate in enumerate(candidates):
            candidate_rows.append({
                "episode": episode.get("episode"),
                "task_key": episode.get("task_key"),
                "success": bool(episode.get("success")),
                "step": step.get("step", step_position),
                "sample": candidate.get("sample", candidate_position),
                "selected": candidate.get("sample", candidate_position) == step.get("selected_sample"),
                "model_action": candidate.get("model_action"),
                "model_decision": candidate.get("model_decision"),
                "action_was_admissible": candidate.get("action_was_admissible"),
                "parse_ok": candidate.get("parse_ok"),
                "parse_errors": candidate.get("parse_errors") or [],
                "distill_eligible": candidate.get("distill_eligible"),
                "distill_exclusion_reasons": candidate.get("distill_exclusion_reasons") or [],
            })

episode_df = pd.DataFrame(episode_rows).sort_values("episode").reset_index(drop=True)
step_df = pd.DataFrame(step_rows)
candidate_df = pd.DataFrame(candidate_rows)
episode_lookup = {record["episode"]: record for record in episodes}

print(f"Loaded {len(episode_df):,} episodes, {len(step_df):,} steps, and {len(candidate_df):,} candidates.")

## Dataset overview

In [ ]:
overview = pd.Series({
    "episodes": len(episode_df),
    "successful": int(episode_df["success"].sum()),
    "success_rate": episode_df["success"].mean(),
    "total_steps": int(episode_df["n_steps"].sum()),
    "median_steps": episode_df["n_steps"].median(),
    "fallback_steps": int(episode_df["fallback_steps"].sum()),
}, name="value").to_frame()
display(overview.style.format({"value": lambda value: f"{value:.1%}" if 0 < value < 1 else f"{value:,.0f}"}))

task_summary = (
    episode_df.groupby("task_key", dropna=False)
    .agg(episodes=("episode", "size"), success_rate=("success", "mean"),
         median_steps=("n_steps", "median"), max_steps=("n_steps", "max"))
    .sort_values("success_rate")
)
display(task_summary.style.format({"success_rate": "{:.1%}", "median_steps": "{:.1f}"}))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.barplot(data=task_summary.reset_index(), x="task_key", y="success_rate", ax=axes[0], color="#4C78A8")
axes[0].set(title="Success rate by task", xlabel="Task", ylabel="Success rate", ylim=(0, 1.05))
axes[0].yaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
sns.histplot(data=episode_df, x="n_steps", hue="success", multiple="stack", discrete=True, ax=axes[1])
axes[1].set(title="Episode length", xlabel="Steps", ylabel="Episodes")
plt.tight_layout()
plt.show()

## Select episodes

Filters are combined with AND. Leave a filter empty or `None` to disable it. Explicit `EPISODE_IDS` are also filtered by the other settings. The defaults compare one successful and one failed trajectory from the sample.

In [ ]:
# --- Edit these values ---
EPISODE_IDS: list[int] = [0, 7]     # [] means any episode
TASK_KEYS: list[str] = []            # e.g. ["cool", "heat"]
SUCCESS: bool | None = None          # True, False, or None
DESCRIPTION_CONTAINS = ""          # case-insensitive substring
MIN_STEPS: int | None = None
MAX_STEPS: int | None = None
# -------------------------

selected_df = episode_df.copy()
if EPISODE_IDS:
    selected_df = selected_df[selected_df["episode"].isin(EPISODE_IDS)]
if TASK_KEYS:
    selected_df = selected_df[selected_df["task_key"].isin(TASK_KEYS)]
if SUCCESS is not None:
    selected_df = selected_df[selected_df["success"].eq(SUCCESS)]
if DESCRIPTION_CONTAINS:
    selected_df = selected_df[selected_df["task_description"].fillna("").str.contains(
        DESCRIPTION_CONTAINS, case=False, regex=False
    )]
if MIN_STEPS is not None:
    selected_df = selected_df[selected_df["n_steps"].ge(MIN_STEPS)]
if MAX_STEPS is not None:
    selected_df = selected_df[selected_df["n_steps"].le(MAX_STEPS)]

selected_ids = selected_df["episode"].tolist()
if not selected_ids:
    print("No episodes matched. Relax the filters above.")
else:
    display(selected_df[["episode", "task_key", "success", "n_steps", "fallback_steps",
                         "retry_count", "task_description"]].style
            .format({"success": lambda value: "yes" if value else "no"})
            .hide(axis="index"))

## Read individual trajectories

`show_episode` displays the initial state followed by one row per action. Long text wraps in the table. Set `show_decisions=False` for a compact action/observation transcript.

In [ ]:
def _clip(value: Any, max_chars: int | None) -> Any:
    if value is None or max_chars is None:
        return value
    text = str(value)
    return text if len(text) <= max_chars else text[: max_chars - 1] + "…"


def show_episode(episode_id: int, *, show_decisions: bool = True, show_candidates: bool = False,
                 max_chars: int | None = 500) -> pd.DataFrame:
    if episode_id not in episode_lookup:
        raise KeyError(f"Unknown episode {episode_id}. Available range: {min(episode_lookup)}–{max(episode_lookup)}")

    episode = episode_lookup[episode_id]
    status = "✅ success" if episode.get("success") else "❌ failure"
    display(Markdown(
        f"### Episode {episode_id} — {status}\n\n"
        f"**Task:** `{episode.get('task_key', 'unknown')}` — {episode.get('task_description', '')}  \n"
        f"**Steps:** {len(episode.get('steps', []))}  \n"
        f"**Initial observation:** {episode.get('initial_observation', '')}"
    ))

    rows = []
    for position, step in enumerate(episode.get("steps", [])):
        row = {
            "step": step.get("step", position),
            "action": _clip(step.get("action_taken", step.get("action")), max_chars),
            "observation": _clip(step.get("observation"), max_chars),
            "attempts": step.get("sample_attempts"),
            "retries": step.get("empty_action_retries"),
            "fallback": bool(step.get("fallback_used")),
        }
        if show_decisions:
            row["decision"] = _clip(step.get("selected_model_decision", step.get("decision")), max_chars)
        rows.append(row)

    trajectory = pd.DataFrame(rows)
    display(trajectory.style.set_properties(**{"white-space": "pre-wrap", "text-align": "left"}).hide(axis="index"))

    if show_candidates:
        candidates = candidate_df[candidate_df["episode"].eq(episode_id)].copy()
        display(Markdown("#### Candidate samples"))
        display(candidates.style.set_properties(**{"white-space": "pre-wrap", "text-align": "left"}).hide(axis="index"))
    return trajectory


for episode_id in selected_ids:
    show_episode(episode_id, show_decisions=True, show_candidates=False, max_chars=500)

## Compare selected episodes

The first plot shows how actions accumulate over time. The second flags steps that had retries, fallbacks, inadmissible samples, parse failures, or distillation exclusions.

In [ ]:
selected_steps = step_df[step_df["episode"].isin(selected_ids)].copy()
selected_candidates = candidate_df[candidate_df["episode"].isin(selected_ids)].copy()

if not selected_ids:
    print("Select at least one episode above.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, max(4, 0.45 * len(selected_ids) + 2)))
    sns.scatterplot(data=selected_steps, x="step", y="episode", hue="success",
                    palette={True: "#54A24B", False: "#E45756"}, s=55, ax=axes[0])
    axes[0].set(title="Recorded steps by episode", xlabel="Step", ylabel="Episode")

    diagnostics = []
    for episode_id in selected_ids:
        eps = selected_steps[selected_steps["episode"].eq(episode_id)]
        cand = selected_candidates[selected_candidates["episode"].eq(episode_id)]
        diagnostics.append({
            "episode": episode_id,
            "retries": int(eps["empty_action_retries"].fillna(0).sum()),
            "fallbacks": int(eps["fallback_used"].sum()),
            "inadmissible": int(cand["action_was_admissible"].eq(False).sum()),
            "parse failures": int(cand["parse_ok"].eq(False).sum()),
            "distill exclusions": int(cand["distill_eligible"].eq(False).sum()),
        })
    diagnostic_df = pd.DataFrame(diagnostics).set_index("episode")
    sns.heatmap(diagnostic_df, annot=True, fmt="g", cmap="Reds", cbar=False, linewidths=0.5, ax=axes[1])
    axes[1].set(title="Trajectory diagnostics", xlabel="Event count", ylabel="Episode")
    plt.tight_layout()
    plt.show()
    display(diagnostic_df)

## Action patterns

Actions are grouped by their first verb to make loops and policy differences easier to spot.

In [ ]:
def action_verb(action: Any) -> str:
    if not isinstance(action, str) or not action.strip():
        return "<empty>"
    return action.strip().split(maxsplit=1)[0].lower()


if selected_steps.empty:
    print("Select at least one episode above.")
else:
    action_counts = (selected_steps.assign(action_verb=selected_steps["action"].map(action_verb))
                     .groupby(["episode", "action_verb"]).size().rename("count").reset_index())
    pivot = action_counts.pivot(index="episode", columns="action_verb", values="count").fillna(0).astype(int)
    display(pivot.style.background_gradient(cmap="Blues", axis=None))

    repeated = []
    for episode_id, group in selected_steps.sort_values(["episode", "step"]).groupby("episode"):
        actions = group["action"].fillna("<empty>").tolist()
        runs = []
        start = 0
        for index in range(1, len(actions) + 1):
            if index == len(actions) or actions[index] != actions[start]:
                if index - start > 1:
                    runs.append({"episode": episode_id, "start_step": int(group.iloc[start]["step"]),
                                 "run_length": index - start, "action": actions[start]})
                start = index
        repeated.extend(runs)
    display(Markdown("**Consecutive repeated-action runs**"))
    display(pd.DataFrame(repeated) if repeated else Markdown("No consecutive repeated actions in the selection."))

## Search for episodes

Search across task descriptions, initial observations, decisions, actions, and observations. By default the query is treated as plain text; pass `regex=True` for a regular expression.

In [ ]:
def search_episodes(query: str, *, regex: bool = False, case: bool = False) -> pd.DataFrame:
    if not query:
        return episode_df.iloc[0:0].copy()
    pattern = re.compile(query if regex else re.escape(query), 0 if case else re.IGNORECASE)
    matches = []
    for episode in episodes:
        fields: list[tuple[str, Any]] = [
            ("task_description", episode.get("task_description")),
            ("initial_observation", episode.get("initial_observation")),
        ]
        for position, step in enumerate(episode.get("steps", [])):
            step_number = step.get("step", position)
            fields.extend([
                (f"step {step_number} decision", step.get("selected_model_decision", step.get("decision"))),
                (f"step {step_number} action", step.get("action_taken", step.get("action"))),
                (f"step {step_number} observation", step.get("observation")),
            ])
        hit_locations = [label for label, value in fields if value is not None and pattern.search(str(value))]
        if hit_locations:
            matches.append({
                "episode": episode.get("episode"), "task_key": episode.get("task_key"),
                "success": bool(episode.get("success")), "n_steps": len(episode.get("steps", [])),
                "task_description": episode.get("task_description"),
                "match_count": len(hit_locations), "first_match": hit_locations[0],
            })
    return pd.DataFrame(matches).sort_values(["match_count", "episode"], ascending=[False, True]) if matches else pd.DataFrame()


# Example: find every episode mentioning a microwave.
search_episodes("microwave").head(20)

## Inspect candidate-level failures

This view keeps only candidates with a parse error, inadmissible action, or distillation exclusion.

In [ ]:
problem_candidates = selected_candidates[
    selected_candidates["parse_ok"].eq(False)
    | selected_candidates["action_was_admissible"].eq(False)
    | selected_candidates["distill_eligible"].eq(False)
].copy()

problem_columns = ["episode", "step", "sample", "selected", "model_action",
                   "action_was_admissible", "parse_ok", "parse_errors",
                   "distill_eligible", "distill_exclusion_reasons"]
if problem_candidates.empty:
    display(Markdown("No candidate-level problems in the selected episodes."))
else:
    display(problem_candidates[problem_columns].style
            .set_properties(**{"white-space": "pre-wrap", "text-align": "left"})
            .hide(axis="index"))

## Export the selection (optional)

The function writes the original nested episode records plus normalized summary and step CSVs. Nothing is written until you call it.

In [ ]:
def export_selection(output_dir: str | Path = "episode_analysis_export") -> Path:
    if not selected_ids:
        raise ValueError("The current selection is empty")
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    selected_records = [episode_lookup[episode_id] for episode_id in selected_ids]
    (output_dir / "episodes.json").write_text(json.dumps(selected_records, indent=2), encoding="utf-8")
    selected_df.to_csv(output_dir / "episode_summary.csv", index=False)
    selected_steps.to_csv(output_dir / "steps.csv", index=False)
    selected_candidates.to_csv(output_dir / "candidates.csv", index=False)
    return output_dir.resolve()


# Uncomment when you want files:
# export_selection()